<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m1_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 · Notebook 3 — Pandas for Bioengineers
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** Pandas reference — you do **not** need any other notebook. It covers everything in
Géron's `tools_pandas`, re-ordered around the "Get data → Explore → Prepare" workflow of his Chapter 2,
and worked on **real biomedical tables**: the Wisconsin breast-cancer diagnostic dataset (bundled in
Scikit-Learn, offline) with an optional path to **Broad Institute DepMap/CCLE** data.

Run in **Google Colab** (*Runtime → Run all*). The core is fully offline; one optional cell fetches real
omics data and is guarded so the notebook never breaks.

**Contents**
1. Why Pandas — labelled arrays
2. Series and DataFrame
3. Loading a real dataset
4. Looking first: `head`, `info`, `describe`
5. Selecting: `[]`, `.loc`, `.iloc`, boolean filters
6. New & transformed columns
7. Missing data
8. Split–apply–combine: `groupby`
9. Sorting, counting, pivoting
10. Combining tables: `merge`, `concat`
11. Pandas → NumPy for the models
12. Capstone EDA + optional Broad DepMap data

Most sections end with an **Exercise**; run the **Solution** cell to check.

---
*Attribution: adapted from Aurélien Géron's `tools_pandas.ipynb` ([github.com/ageron/handson-mlp](https://github.com/ageron/handson-mlp)), © Aurélien Géron, licensed under the [Apache License 2.0](https://github.com/ageron/handson-mlp/blob/main/LICENSE). Modified: reordered, extended with biomedical examples and exercises, and adapted for this course.*

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 8)   # keep wide tables readable
print("Pandas", pd.__version__)

---
## 1 · Why Pandas — labelled arrays

NumPy arrays are unlabelled and single-typed. Real data arrives as a **table** with named columns of
mixed types and missing entries. **Pandas** adds *labels* (an index on rows and columns) and
*alignment* on top of NumPy — it is where every ML project begins, and what feeds clean arrays to the
models.

---
## 2 · Series and DataFrame

A **Series** is a 1-D labelled array (values + an index). A **DataFrame** is a 2-D table: ordered
columns, each a Series, sharing one row index. Think "NumPy array with named rows and columns whose
columns may differ in type.

In [ ]:
s = pd.Series([72, 80, 68], index=["A", "B", "C"], name="heart_rate")
print(s, "\n")
df = pd.DataFrame({
    "age":     [58, 71, 45],
    "glucose": [131.4, 99.0, 152.7],
    "dx":      ["benign", "benign", "malignant"],
}, index=["A", "B", "C"])
print(df)
print("\nshape:", df.shape, "| columns:", list(df.columns), "| dtypes:\n", df.dtypes)

---
## 3 · Loading a real dataset

You normally load a CSV with `pd.read_csv("file.csv")`. Here we use the **Wisconsin breast-cancer**
dataset bundled in Scikit-Learn (no download, works offline): 569 biopsies × 30 numeric features, with a
malignant/benign label.

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame                                  # features + a 'target' column
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})
print("shape:", df.shape)
df.head()

---
## 4 · Looking first: `head`, `info`, `describe`

Always inspect before analysing: `head()` shows the top rows, `info()` the columns/types/missingness,
`describe()` the numeric summary.

In [ ]:
df.info()

In [ ]:
df[["mean radius", "mean texture", "mean area"]].describe()

---
## 5 · Selecting: `[]`, `.loc`, `.iloc`, boolean filters

`df["col"]` picks a column. **`.loc`** selects by *label*, **`.iloc`** by *integer position* — mixing
them up is the classic beginner error. Boolean arrays filter rows.

In [ ]:
print(df["mean radius"].head(3))            # a column (Series)
print("\n.loc by label:\n", df.loc[0, ["mean radius", "diagnosis"]])
print("\n.iloc by position:\n", df.iloc[0, :3])
# boolean filtering: large, malignant tumours
big_mal = df[(df["diagnosis"] == "malignant") & (df["mean radius"] > 20)]
print("\nlarge malignant rows:", len(big_mal))

**Exercise 5.** Select all rows whose `mean area` exceeds 1500, returning only the columns
`mean area` and `diagnosis`. How many are there, and what is their diagnosis mix?

In [ ]:
# Solution 5
sel = df.loc[df["mean area"] > 1500, ["mean area", "diagnosis"]]
print("count:", len(sel))
print(sel["diagnosis"].value_counts())

---
## 6 · New & transformed columns

Create columns with vectorised expressions (fast, like NumPy) or `.apply`/`.map` for elementwise logic.

In [ ]:
# A vectorised derived feature: area-to-perimeter ratio
df["compactness_check"] = df["mean area"] / df["mean perimeter"]
# map a continuous column to a category with apply
df["size_class"] = df["mean radius"].apply(lambda r: "large" if r > 15 else "small")
print(df[["mean radius", "size_class", "compactness_check"]].head())

---
## 7 · Missing data

Real tables have holes, shown as `NaN`. Find them with `isna()`, handle with `dropna`/`fillna`. (This
dataset is complete; we inject a hole to demonstrate.)

In [ ]:
df2 = df.copy()
df2.loc[0:2, "mean texture"] = np.nan           # inject missing values
print("missing per column (top 3):\n", df2.isna().sum().sort_values(ascending=False).head(3))
print("\nfill with the column median:")
df2["mean texture"] = df2["mean texture"].fillna(df2["mean texture"].median())
print("missing after fill:", int(df2["mean texture"].isna().sum()))

---
## 8 · Split–apply–combine: `groupby`

`groupby` splits rows into groups, applies a function to each, and combines the results — the natural way
to compute per-group statistics (e.g. a mean biomarker per diagnosis).

In [ ]:
summary = df.groupby("diagnosis")[["mean radius", "mean area", "mean concavity"]].mean()
print(summary)
print("\ncounts per class:\n", df.groupby("diagnosis").size())
# multiple statistics at once
print("\nradius stats by class:\n", df.groupby("diagnosis")["mean radius"].agg(["mean", "std", "min", "max"]))

**Exercise 8.** Compute the mean of **every** feature per diagnosis, then report which diagnosis has
the larger mean `worst area`.

In [ ]:
# Solution 8
means = df.groupby("diagnosis")[data.feature_names].mean()
print(means["worst area"])
print("larger worst area:", means["worst area"].idxmax())

---
## 9 · Sorting, counting, pivoting

`sort_values` orders rows; `value_counts` tallies categories; `pivot_table` cross-tabulates.

In [ ]:
print("top 3 by mean area:\n",
      df.sort_values("mean area", ascending=False)[["mean area", "diagnosis"]].head(3))
print("\nclass balance:\n", df["diagnosis"].value_counts(normalize=True).round(3))
print("\npivot: mean radius by diagnosis x size_class:\n",
      df.pivot_table(values="mean radius", index="diagnosis", columns="size_class", aggfunc="mean").round(2))

---
## 10 · Combining tables: `merge`, `concat`

`concat` stacks tables; `merge` joins them on a key (a database-style join) — common when features and
labels, or two assays, live in separate files.

In [ ]:
labels = pd.DataFrame({"sample": [0, 1, 2], "site": ["L", "R", "L"]})
feats  = pd.DataFrame({"sample": [0, 1, 2], "mean radius": df["mean radius"].head(3).values})
merged = feats.merge(labels, on="sample")        # join on the shared key
print(merged)
print("\nconcat two row-blocks:\n", pd.concat([feats.head(1), feats.tail(1)]))

---
## 11 · Pandas → NumPy for the models

Scikit-Learn and PyTorch want plain NumPy arrays. `to_numpy()` (or `.values`) extracts the underlying
matrix; this is the bridge from the table back to Notebook 2's array world.

In [ ]:
X = df[data.feature_names].to_numpy().astype("float32")   # (569, 30)
y = df["target"].to_numpy()
print("X:", X.shape, X.dtype, "| y:", y.shape, "| classes:", np.unique(y))
# standardise with broadcasting (Notebook 2) -- exactly what StandardScaler does
Xs = (X - X.mean(axis=0)) / X.std(axis=0)
print("standardised feature means ~0:", np.round(Xs.mean(axis=0)[:3], 6))

---
## 12 · Capstone EDA, and optional Broad DepMap/CCLE data

You now have the full "Get → Explore → Prepare" loop of Géron's Chapter 2: we loaded a real diagnostic
table, inspected it, engineered a feature, summarised by group, and produced a clean standardised matrix
for modelling.

**Optional — real Broad Institute omics.** The cell below shows the workflow for **DepMap/CCLE** cancer
cell-line data. The full files are large, so download a small slice from the portal
(`https://depmap.org/portal/download/`, e.g. a drug-sensitivity or expression CSV) into Colab, then read
it with `pd.read_csv`. The cell is guarded so it runs (with instructions) whether or not the file is
present.

In [ ]:
# Optional: real Broad DepMap/CCLE data (download a CSV from depmap.org/portal/download first)
try:
    ccle = pd.read_csv("CCLE_sample.csv")        # <- your downloaded slice
    print("Loaded DepMap/CCLE slice:", ccle.shape)
    print(ccle.head())
except FileNotFoundError:
    print("No local DepMap file found (this is fine).")
    print("To use real Broad data: download a small CSV from")
    print("  https://depmap.org/portal/download/   (e.g. drug sensitivity or expression),")
    print("  upload it to Colab, then:  ccle = pd.read_csv('your_file.csv')")
    print("Everything above used the offline breast-cancer dataset and needs no download.")

**Exercise 12.** Build a one-row-per-class summary table containing, for each diagnosis: the sample
count, and the mean of `mean radius`, `mean area` and `mean concavity`. (One `groupby` + `agg`.)

In [ ]:
# Solution 12
out = df.groupby("diagnosis").agg(
    n=("diagnosis", "size"),
    mean_radius=("mean radius", "mean"),
    mean_area=("mean area", "mean"),
    mean_concavity=("mean concavity", "mean"),
).round(2)
print(out)

---
### You now know Pandas
Series and DataFrames, label vs position selection, derived columns, missing-data handling,
split–apply–combine, joins, and the hand-off to NumPy. **Next:** Notebook 4 (Matplotlib) visualises these
tables and arrays; Module 2 develops the linear algebra behind the models you will fit to this `X`.